# Hindi Song Chunking and Annotation Pipeline

This notebook breaks songs into chunks, transcribes Hindi lyrics with Faster-Whisper, estimates BPM/key, generates audio captions with Gemini, and saves `.mp3`, `.lyrics.txt`, `.caption.txt`, and `.json` outputs.

**Run one section at a time.** The Gemini test is intentionally separate from Whisper so you can verify API access before running the expensive model.

## Before you start
1. Select the `.venv` Python interpreter in VS Code and choose **Run All Above** / run cells sequentially.
2. Install dependencies in that environment if needed: `pip install numpy librosa faster-whisper google-genai nbformat`
3. Install FFmpeg and ensure `ffmpeg` works in the VS Code terminal.
4. Set `GEMINI_API_KEY` in the environment before starting Jupyter. Do not paste your API key into the notebook.
5. The original code uses CUDA. If you do not have a compatible NVIDIA CUDA GPU, use the CPU configuration in the Whisper cell.


## 1. Imports and configuration

In [12]:
import os
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import librosa
from google import genai
from google.genai import types

# Change these paths if your project is stored elsewhere.
SRC = Path(r"GACMIS\Dataset\Songs")
DST = Path(r"GACMIS\Dataset\Chunks")

TARGET = 42.0       # desired chunk duration, seconds
SNAP = 3.0          # search +/- this many seconds for a low-energy cut
MIN_TAIL = 20.0     # avoid leaving a very short final chunk
GEMINI_MODEL = "gemini-2.5-flash"
EXTS = {".mp3", ".wav", ".flac", ".ogg", ".opus", ".m4a"}

DST.mkdir(parents=True, exist_ok=True)

print("Source:", SRC.resolve())
print("Source exists:", SRC.exists())
print("Destination:", DST.resolve())
print("Destination ready:", DST.exists())


Source: C:\Users\rdips\OneDrive\Documents\CS787proj\GACMIS\Dataset\Songs
Source exists: True
Destination: C:\Users\rdips\OneDrive\Documents\CS787proj\GACMIS\Dataset\Chunks
Destination ready: True


In [2]:
import os

FFMPEG = (
    r"C:\Users\rdips\AppData\Local\Microsoft\WinGet\Packages"
    r"\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe"
    r"\ffmpeg-9.0.2-full_build\bin\ffmpeg.exe"
)

os.environ["PATH"] = (
    os.path.dirname(FFMPEG)
    + os.pathsep
    + os.environ.get("PATH", "")
)

print("FFmpeg configured:", os.path.isfile(FFMPEG))

FFmpeg configured: True


## 2. Check FFmpeg

This is needed to export each chunk as an MP3. If this cell fails, install FFmpeg and add it to PATH before continuing.

In [3]:
result = subprocess.run(
    ["ffmpeg", "-version"],
    capture_output=True,
    text=True
)
print("Return code:", result.returncode)
print((result.stdout or result.stderr).splitlines()[0] if (result.stdout or result.stderr) else "No output")
if result.returncode != 0:
    raise RuntimeError("FFmpeg is unavailable. Install it and ensure `ffmpeg` is on PATH.")


Return code: 0
ffmpeg version 9.0.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers


In [4]:
import os
import subprocess
from pathlib import Path

ffmpeg_path = Path(
    r"C:\Users\rdips\AppData\Local\Microsoft\WinGet\Packages"
    r"\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe"
    r"\ffmpeg-9.0.2-full_build\bin\ffmpeg.exe"
)

print("FFmpeg exists:", ffmpeg_path.exists())

result = subprocess.run(
    [str(ffmpeg_path), "-version"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout[:500])

FFmpeg exists: True
ffmpeg version 9.0.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
built with gcc 16.2.0 (Rev3, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisy


## 3. Find audio files

Run this before loading Whisper. If zero files are found, correct `SRC`.

In [5]:
songs = sorted(
    p for p in SRC.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTS
) if SRC.exists() else []

print(f"Found {len(songs)} audio file(s).")
for p in songs[:10]:
    print(" -", p)
if not songs:
    print("No audio files found. Check the SRC path and supported extensions.")


Found 363 audio file(s).
 - GACMIS\Dataset\Songs\Bollywood_Rap\Aditi Singh Sharma - High Rated Gabru (Female Version).opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Aditi Singh Sharma - High Rated Gabru (Female Version).test.wav
 - GACMIS\Dataset\Songs\Bollywood_Rap\Amaal Mallik - Zero Hour Mashup 2015(Remix By Dj Kiran Kamath).opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Anirudh Ravichander - Pori Pathi Vizhum.opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Badshah - Proper Patola.opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Badshah - She Move It Like.opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Badshah - Tere Naal Nachna.opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Big Dhillon - Stand Up - Spoken Word.opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Bohemia - Aja Ni Aja.opus
 - GACMIS\Dataset\Songs\Bollywood_Rap\Bohemia - Beparwah.opus


## 4. Test the Gemini API key and text-only request

This tests authentication/model access without sending audio and without loading Whisper. If this fails, fix the API key, project, quota, or model issue before continuing.

In [1]:

# 4. Test the Gemini API key and text-only request

%pip install -U google-genai

import getpass
from google import genai

# Enter your API key securely
api_key = getpass.getpass("Enter your Gemini API key: ").strip()

if not api_key:
    raise ValueError("API key cannot be empty.")

# Initialize the client
client = genai.Client(api_key=api_key)

# Test text-only access without loading Whisper or audio
print("Testing Gemini API access...")

try:
    interaction = client.interactions.create(
        model="gemini-3.8-flash",
        input="Reply with exactly: GEMINI API TEST SUCCESSFUL"
    )

    print("\nSUCCESS — Gemini API is working!")
    print("Response:", interaction.output_text)

except Exception as e:
    print("\nFAILED — Gemini API test did not complete.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))


Note: you may need to restart the kernel to use updated packages.
Testing Gemini API access...

SUCCESS — Gemini API is working!
Response: GEMINI API TEST SUCCESSFUL


## 5. Test Gemini with one audio file

Set `TEST_AUDIO` to an existing, small audio file. This tests the same audio-caption request used later, independently of the full pipeline.

In [31]:

from pathlib import Path
from google.genai import types

# Use the same client and model from your successful Gemini API test.
# If your test uses a different model variable, set GEMINI_MODEL accordingly.
GEMINI_MODEL = globals().get("GEMINI_MODEL", "gemini-2.5-flash")


def caption_audio(audio_path, genre):
    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    if "client" not in globals():
        raise RuntimeError(
            "Gemini client is not defined. Run your successful "
            "Gemini API initialization cell first."
        )

    print(f"[GEMINI] Uploading {audio_path.name}...", flush=True)

    uploaded_file = client.files.upload(
        file=str(audio_path),
        config=types.UploadFileConfig(mime_type="audio/mpeg"),
    )

    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                uploaded_file,
                f"""
                Listen carefully to this music audio segment.

                Genre/category: {genre}

                Write a concise but informative audio caption describing:
                - Whether vocals are present and the vocal style
                - The language, if identifiable
                - Instruments and accompaniment
                - Rhythm, tempo feel, and mood
                - Production and musical characteristics

                Describe only what is audible. Do not invent lyrics,
                instruments, or musical details. Return only the caption.
                """,
            ],
        )

        caption = (response.text or "").strip()

        if not caption:
            raise RuntimeError("Gemini returned an empty caption.")

        print(f"[GEMINI] Caption generated for {audio_path.name}", flush=True)
        return caption

    finally:
        # Clean up the uploaded temporary Gemini file when possible.
        try:
            client.files.delete(name=uploaded_file.name)
        except Exception as cleanup_error:
            print(
                f"[GEMINI WARNING] Could not delete uploaded file: "
                f"{cleanup_error}",
                flush=True,
            )


print("caption_audio defined:", callable(caption_audio))
print("Gemini model:", GEMINI_MODEL)

caption_audio defined: True
Gemini model: gemini-2.5-flash


In [8]:
from pathlib import Path

DATASET_DIR = Path("GACMIS/Dataset/Songs")

AUDIO_EXTENSIONS = {".mp3", ".opus", ".wav", ".m4a", ".flac", ".ogg"}

songs = sorted(
    p for p in DATASET_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
)

print(f"Found {len(songs)} audio files.")

if not songs:
    raise FileNotFoundError(
        f"No audio files found. Check the dataset path: {DATASET_DIR.resolve()}"
    )

TEST_AUDIO = songs[0]

print("Test audio:", TEST_AUDIO)
print("File size (MB):", round(TEST_AUDIO.stat().st_size / (1024 * 1024), 2))

Found 363 audio files.
Test audio: GACMIS\Dataset\Songs\Bollywood_Rap\Aditi Singh Sharma - High Rated Gabru (Female Version).opus
File size (MB): 2.06


In [3]:

import time
from pathlib import Path
from google import genai
from google.genai import types

# Use the same API key you tested successfully.
# If client already exists, you can reuse it.
GEMINI_MODEL = "gemini-3.8-flash"

def gemini_caption(path, prompt):
    path = Path(path)

    if not path.is_file():
        print(f"[Gemini] Audio file not found: {path}")
        return ""

    # Your dataset contains .opus files as well as MP3s.
    # Use the correct MIME type for each format.
    suffix = path.suffix.lower()
    mime_type = {
        ".mp3": "audio/mpeg",
        ".opus": "audio/ogg",
        ".wav": "audio/wav",
        ".m4a": "audio/mp4",
        ".flac": "audio/flac",
    }.get(suffix)

    if mime_type is None:
        print(f"[Gemini] Unsupported audio format: {suffix}")
        return ""

    part = types.Part.from_bytes(
        data=path.read_bytes(),
        mime_type=mime_type
    )

    for attempt in range(4):
        try:
            print(
                f"[Gemini] Sending request "
                f"(attempt {attempt + 1}/4): {path.name}"
            )

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=[part, prompt]
            )

            caption = (response.text or "").strip()

            if caption:
                print("[Gemini] Caption generated successfully.")
                return caption

            print("[Gemini] Empty response received.")

        except Exception as e:
            print(f"[Gemini] Error: {type(e).__name__}: {e}")

        if attempt < 3:
            time.sleep(5 * (attempt + 1))

    print("[Gemini] All attempts failed.")
    return ""


In [4]:

test_path = Path(
    r"GACMIS\Dataset\Songs\Bollywood_Rap\Aditi Singh Sharma - High Rated Gabru (Female Version).opus"
)

test_prompt = (
    "Describe the music, vocals, instruments, and mood of this audio clip "
    "in 2-3 sentences."
)

caption = gemini_caption(test_path, test_prompt)
print("\nFinal caption:", caption)


[Gemini] Sending request (attempt 1/4): Aditi Singh Sharma - High Rated Gabru (Female Version).opus
[Gemini] Caption generated successfully.

Final caption: This track features lively and vibrant female vocals singing an upbeat Punjabi pop song. Driven by a punchy electronic beat, groovy synths, and rhythmic claps, the instrumentation creates a danceable, modern pop groove. The overall mood is energetic, playful, and infectious.


## 6. Load one song and inspect its duration / waveform

This checks librosa/audio decoding before running the rest of the pipeline.

In [9]:

from pathlib import Path
import subprocess
import librosa
import numpy as np

TEST_SONG = Path(songs[0])

# Convert the OPUS/audio file to WAV using FFmpeg
WAV_PATH = TEST_SONG.with_suffix(".test.wav")

FFMPEG = (
    r"C:\Users\rdips\AppData\Local\Microsoft\WinGet\Packages"
    r"\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe"
    r"\ffmpeg-9.0.2-full_build\bin\ffmpeg.exe"
)

if not Path(FFMPEG).is_file():
    raise FileNotFoundError(f"FFmpeg not found: {FFMPEG}")

subprocess.run(
    [
        FFMPEG,
        "-y",
        "-i", str(TEST_SONG),
        "-vn",
        "-ac", "1",
        "-ar", "22050",
        str(WAV_PATH),
    ],
    check=True,
    capture_output=True,
    text=True,
)

# Load the converted WAV file
y, sr = librosa.load(WAV_PATH, sr=22050, mono=True)

duration = len(y) / sr

print("Song:", TEST_SONG.name)
print("Audio loaded successfully!")
print("Sample rate:", sr)
print("Duration:", round(duration, 2), "seconds")
print("Samples:", len(y))
print("Array shape:", y.shape)


Song: Aditi Singh Sharma - High Rated Gabru (Female Version).opus
Audio loaded successfully!
Sample rate: 22050
Duration: 123.19 seconds
Samples: 2716417
Array shape: (2716417,)


In [6]:
%pip install -U jupyter ipywidgets


Note: you may need to restart the kernel to use updated packages.


## 7. Detect chunk boundaries

In [13]:
def find_cuts(y, sr, target=TARGET, snap=SNAP, min_tail=MIN_TAIL):
    dur = len(y) / sr
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    t = librosa.frames_to_time(
        np.arange(len(rms)), sr=sr, hop_length=hop
    )

    cuts, pos = [0.0], 0.0
    while dur - pos > target + min_tail:
        c = pos + target
        mask = (t > c - snap) & (t < c + snap)

        # Guard against an empty search window for unusually short audio.
        if not np.any(mask):
            break

        candidate_times = t[mask]
        candidate_rms = rms[mask]
        c = float(candidate_times[np.argmin(candidate_rms)])

        # Ensure the cut progresses and leaves a non-empty chunk.
        if c <= pos or c >= dur:
            break

        cuts.append(c)
        pos = c

    if cuts[-1] != dur:
        cuts.append(float(dur))
    return cuts

cuts = find_cuts(y, sr)
print("Cut times (seconds):", [round(c, 2) for c in cuts])
print("Chunk durations:", [round(cuts[i+1] - cuts[i], 2) for i in range(len(cuts)-1)])


Cut times (seconds): [0.0, 40.59, 81.83, 123.19]
Chunk durations: [40.59, 41.24, 41.37]


In [14]:
for name in ("TARGET", "SNAP", "MIN_TAIL", "y", "sr"):
    print(f"{name}: {globals().get(name, 'NOT DEFINED')}")

TARGET: 42.0
SNAP: 3.0
MIN_TAIL: 20.0
y: [0. 0. 0. ... 0. 0. 0.]
sr: 22050


## 8. Calculate BPM and estimated key

Key detection uses correlations against major/minor key profiles, so treat the result as an estimate, not ground truth.

In [15]:
MAJ = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
MIN = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
NOTES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

def bpm_key(y, sr):
    if len(y) < sr:
        raise ValueError("Audio segment is too short for reliable BPM/key estimation.")

    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    bpm = int(round(float(np.atleast_1d(tempo)[0])))

    chroma = librosa.feature.chroma_cqt(y=y, sr=sr).mean(axis=1)
    best, name = -2, "C major"

    for i in range(12):
        for profile, mode in ((MAJ, "major"), (MIN, "minor")):
            score = np.corrcoef(chroma, np.roll(profile, i))[0, 1]
            if np.isfinite(score) and score > best:
                best, name = score, f"{NOTES[i]} {mode}"

    return bpm, name

test_bpm, test_key = bpm_key(y[:min(len(y), sr * 45)], sr)
print("Estimated BPM:", test_bpm)
print("Estimated key:", test_key)


Estimated BPM: 123
Estimated key: E major


## 9. Load Faster-Whisper

This can download a large model the first time. Use CUDA only if your machine has a compatible NVIDIA GPU and CUDA setup. Otherwise switch to CPU with `device="cpu", compute_type="int8"` (slower).

In [16]:
from faster_whisper import WhisperModel
import time

print("Loading Whisper small on CPU...", flush=True)
start = time.time()

whisper = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print(f"Model loaded in {time.time() - start:.1f} seconds!")

Loading Whisper small on CPU...
Model loaded in 2.3 seconds!


## 10. Test transcription on one song

In [17]:
segments, info = whisper.transcribe(
    str(TEST_SONG),
    language="hi",
    word_timestamps=True,
    condition_on_previous_text=False,
    vad_filter=False
)
segments = list(segments)  # consume the generator so errors occur in this cell
words = [word for segment in segments for word in segment.words]

print("Detected language:", info.language)
print("Number of segments:", len(segments))
print("Number of words:", len(words))
print("First few words:")
for word in words[:20]:
    print(round(word.start, 2), round(word.end, 2), repr(word.word))


Detected language: hi
Number of segments: 9
Number of words: 53
First few words:
90.0 90.3 ' नच्ड़'
90.3 90.48 ' स्द़'
90.48 90.66 ' रथ'
90.66 90.94 ' बी'
90.94 91.16 ' ना'
91.16 92.54 ' याईआ'
93.6 94.2 ' मेर्दे'
94.2 94.96 ' शर्टा'
94.96 96.08 ' लाड़के'
96.98 97.4 ' दूज'
97.4 97.9 ' किन'
97.9 98.06 ' ने'
98.06 98.64 ' और'
98.64 99.1 ' आई'
99.1 99.36 ' आे'
99.36 99.56 ' आे'
99.56 100.16 ' माख्रा'
100.16 101.22 ' मेरानी'
101.78 102.2 ' अे'
102.2 102.74 ' रेडद'


In [48]:
segments, info = whisper.transcribe(
    str(TEST_SONG),
    language="hi",
    word_timestamps=True
)

segments = list(segments)

print("Detected language:", info.language)
print("Duration:", info.duration)
print("Segments:", len(segments))

for segment in segments[:5]:
    print(
        f"{segment.start:.2f}s - "
        f"{segment.end:.2f}s: {segment.text}"
    )

TypeError: open() got an unexpected keyword argument 'metadata_errors'

## 11. Test FFmpeg chunk export

Exports just the first chunk of `TEST_SONG` to a temporary test output. Inspect the file before running the full pipeline.

In [22]:
from pathlib import Path
import subprocess

# Use the full FFmpeg path that you verified earlier
FFMPEG = (
    r"C:\Users\rdips\AppData\Local\Microsoft\WinGet\Packages"
    r"\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe"
    r"\ffmpeg-9.0.2-full_build\bin\ffmpeg.exe"
)

TEST_OUT = DST / "_notebook_test"
TEST_OUT.mkdir(parents=True, exist_ok=True)

# First detected audio chunk
a, b = float(cuts[0]), float(cuts[1])
duration = b - a

test_mp3 = TEST_OUT / f"{TEST_SONG.stem}_test.mp3"

# Physically extract the audio chunk
subprocess.run(
    [
        FFMPEG,
        "-y",
        "-loglevel", "error",
        "-ss", f"{a:.3f}",
        "-i", str(TEST_SONG),
        "-t", f"{duration:.3f}",
        "-vn",
        "-ar", "44100",
        "-b:a", "192k",
        str(test_mp3),
    ],
    check=True,
    capture_output=True,
    text=True,
)

print("Created:", test_mp3.resolve())
print("File exists:", test_mp3.is_file())
print("Expected duration (seconds):", round(duration, 3))
print("Size (KB):", round(test_mp3.stat().st_size / 1024, 1))

Created: C:\Users\rdips\OneDrive\Documents\CS787proj\GACMIS\Dataset\Chunks\_notebook_test\Aditi Singh Sharma - High Rated Gabru (Female Version)_test.mp3
File exists: True
Expected duration (seconds): 40.588
Size (KB): 952.1


In [19]:
from pathlib import Path
import subprocess

print("FFmpeg exists:", Path(FFMPEG).is_file())

result = subprocess.run(
    [FFMPEG, "-version"],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout.splitlines()[0])

FFmpeg exists: True
ffmpeg version 9.0.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers


## 12. Extract words / lyrics for one chunk

In [23]:
def lyrics_for_interval(words, a, b):
    lines, cur, last = [], [], None
    for w in words:
        if a <= w.start < b:
            if last is not None and w.start - last > 0.8 and cur:
                lines.append(" ".join(cur))
                cur = []
            cur.append(w.word.strip())
            last = w.end
    if cur:
        lines.append(" ".join(cur))
    return "\n".join(lines) if lines else "[instrumental]"

test_lyrics = lyrics_for_interval(words, a, b)
print(test_lyrics)


[instrumental]


## 13. Full pipeline

Only run this after the separate tests above pass. It will call Gemini once per chunk that is not skipped, so it can consume API quota. Existing songs are skipped when `<stem>_000.json` already exists; set `SKIP_EXISTING = False` if you intentionally want to regenerate them.

In [36]:
import json
import subprocess
import tempfile
import traceback
import time
from pathlib import Path

import librosa
from google.genai import types


# ============================================================
# 1. GEMINI CAPTION FUNCTION
# ============================================================

# Reuse the Gemini client initialized in your earlier setup cell.
# This sends audio bytes directly; it does NOT use client.files.upload().
GEMINI_MODEL = "gemini-3.8-flash"


def gemini_caption(path, prompt):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Audio file not found: {path}")

    mime_type = {
        ".mp3": "audio/mpeg",
        ".opus": "audio/ogg",
        ".wav": "audio/wav",
        ".m4a": "audio/mp4",
        ".flac": "audio/flac",
        ".ogg": "audio/ogg",
    }.get(path.suffix.lower())

    if mime_type is None:
        raise ValueError(f"Unsupported audio format: {path.suffix}")

    part = types.Part.from_bytes(
        data=path.read_bytes(),
        mime_type=mime_type,
    )

    for attempt in range(4):
        try:
            print(
                f"[Gemini] Sending request "
                f"(attempt {attempt + 1}/4): {path.name}",
                flush=True,
            )

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=[part, prompt],
            )

            caption = (response.text or "").strip()

            if caption:
                print(
                    "[Gemini] Caption generated successfully.",
                    flush=True,
                )
                return caption

            print("[Gemini] Empty response received.", flush=True)

        except Exception as e:
            print(
                f"[Gemini] Error: {type(e).__name__}: {e}",
                flush=True,
            )

        if attempt < 3:
            time.sleep(5 * (attempt + 1))

    return ""


def caption_audio(audio_path, genre):
    prompt = f"""
Describe this music segment from the genre/category: {genre}.

Write a concise but informative caption covering:
- Vocal presence and vocal style
- Language, if identifiable
- Instruments and accompaniment
- Rhythm, tempo feel, and mood
- Production and other audible musical characteristics

Describe only what is audible. Do not invent details.
Return only the caption in 2-3 sentences.
"""

    caption = gemini_caption(audio_path, prompt)

    if not caption:
        raise RuntimeError(
            f"Gemini failed to generate a caption for {audio_path}"
        )

    return caption


# Verify that the Gemini client was initialized earlier.
if "client" not in globals():
    raise NameError(
        "Gemini `client` is not defined. Run your existing cell "
        "that initializes the Google GenAI client, then run this cell."
    )


# ============================================================
# 2. CONFIGURATION: PROCESS ONE SONG ONLY
# ============================================================

SKIP_EXISTING = True

ROMANTIC_DIR = SRC / "Bollywood_Romantic"

AUDIO_EXTENSIONS = {
    ".mp3",
    ".opus",
    ".wav",
    ".m4a",
    ".flac",
    ".ogg",
}

# IMPORTANT: Start with one song. Later, change [:1] to [:25].
romantic_songs = sorted(
    p
    for p in ROMANTIC_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
)[:1]


if not romantic_songs:
    raise FileNotFoundError(
        f"No audio files found in {ROMANTIC_DIR.resolve()}"
    )


print(f"Found {len(romantic_songs)} song(s).", flush=True)
print(f"Input:  {ROMANTIC_DIR.resolve()}", flush=True)
print(f"Output: {DST.resolve()}", flush=True)


# Output structure:
# Chunks/
#   Bollywood_Romantic/
#     song_name_000.mp3
#     song_name_000.lyrics.txt
#     song_name_000.caption.txt
#     song_name_000.json


# ============================================================
# 3. HELPER FUNCTION TO SAVE TEXT
# ============================================================

def write_text(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)

    path.write_text(
        str(content),
        encoding="utf-8",
    )


# ============================================================
# 4. PROCESS ONE SONG
# ============================================================

def process_song(song):

    # Preserve the source folder structure inside DST.
    out = DST / song.parent.relative_to(SRC)
    out.mkdir(parents=True, exist_ok=True)

    stem = song.stem
    genre = song.parent.name.replace("_", " ")

    print(f"\n[SONG] {song.name}", flush=True)

    # --------------------------------------------------------
    # 4A. LOAD THE WHOLE SONG
    # --------------------------------------------------------

    # Convert to WAV temporarily so librosa can load OPUS
    # and other supported FFmpeg input formats reliably.
    try:
        with tempfile.TemporaryDirectory() as temp_dir:

            temp_wav = Path(temp_dir) / "converted.wav"

            subprocess.run(
                [
                    FFMPEG,
                    "-y",
                    "-loglevel", "error",
                    "-i", str(song),
                    "-vn",
                    "-ac", "1",
                    "-ar", "22050",
                    str(temp_wav),
                ],
                check=True,
                capture_output=True,
                text=True,
            )

            y, sr = librosa.load(
                temp_wav,
                sr=22050,
                mono=True,
            )

    except Exception as e:
        print(
            f"[SONG ERROR] Cannot load {song.name}: {e}",
            flush=True,
        )

        return {
            "song": song.name,
            "chunks": 0,
            "errors": 1,
        }

    print(
        f"[AUDIO LOADED] Duration: {len(y) / sr:.2f} seconds",
        flush=True,
    )

    # --------------------------------------------------------
    # 4B. DETECT CHUNK BOUNDARIES
    # --------------------------------------------------------

    # Keep your existing chunk-boundary algorithm unchanged.
    try:
        cuts = find_cuts(y, sr)

        total_chunks = len(cuts) - 1

        print(
            f"[CHUNKS] Detected {total_chunks}",
            flush=True,
        )

    except Exception as e:
        print(
            f"[CUT ERROR] {song.name}: {e}",
            flush=True,
        )

        return {
            "song": song.name,
            "chunks": 0,
            "errors": 1,
        }

    if total_chunks <= 0:
        print(
            f"[CUT ERROR] No valid chunks detected for {song.name}",
            flush=True,
        )

        return {
            "song": song.name,
            "chunks": 0,
            "errors": 1,
        }

    song_errors = 0

    # --------------------------------------------------------
    # 4C. PROCESS EACH CHUNK
    # --------------------------------------------------------

    # Finish attempting every chunk before leaving this song.
    for i in range(total_chunks):

        a = float(cuts[i])
        b = float(cuts[i + 1])

        base = out / f"{stem}_{i:03d}"

        mp3 = Path(f"{base}.mp3")
        lyrics_file = Path(f"{base}.lyrics.txt")
        caption_file = Path(f"{base}.caption.txt")
        json_file = Path(f"{base}.json")

        if b <= a:
            print(
                f"[INVALID BOUNDARIES] {a}, {b}",
                flush=True,
            )

            song_errors += 1
            continue

        # Skip a chunk only if every required file exists
        # and the saved JSON says that processing succeeded.
        if SKIP_EXISTING and json_file.exists():

            try:
                old = json.loads(
                    json_file.read_text(encoding="utf-8")
                )

                if (
                    mp3.exists()
                    and lyrics_file.exists()
                    and caption_file.exists()
                    and old.get("status") == "complete"
                ):
                    print(
                        f"[SKIP] {base.name} already complete",
                        flush=True,
                    )

                    continue

            except Exception:
                pass

        print(
            f"\n[CHUNK {i + 1}/{total_chunks}] "
            f"{a:.2f}s–{b:.2f}s",
            flush=True,
        )

        errors = []
        lyrics = ""
        caption = ""
        bpm = None
        key = None
        audio_ok = False

        # ----------------------------------------------------
        # 4C-1. EXTRACT THE CHUNK AS MP3
        # ----------------------------------------------------

        try:
            print("[FFMPEG] Extracting...", flush=True)

            subprocess.run(
                [
                    FFMPEG,
                    "-y",
                    "-loglevel", "error",
                    "-ss", f"{a:.3f}",
                    "-i", str(song),
                    "-t", f"{b - a:.3f}",
                    "-vn",
                    "-ar", "44100",
                    "-b:a", "192k",
                    str(mp3),
                ],
                check=True,
                capture_output=True,
                text=True,
            )

            if not mp3.exists() or mp3.stat().st_size == 0:
                raise RuntimeError("MP3 missing or empty")

            audio_ok = True

            print(
                f"[OK] MP3: {mp3.name}",
                flush=True,
            )

        except Exception as e:
            errors.append(
                f"audio: {type(e).__name__}: {e}"
            )

            print(
                f"[AUDIO ERROR] {e}",
                flush=True,
            )

        # ----------------------------------------------------
        # 4C-2. TRANSCRIBE THE CHUNK USING WHISPER
        # ----------------------------------------------------

        if audio_ok:

            try:
                print(
                    "[WHISPER] Transcribing...",
                    flush=True,
                )

                segments, _ = whisper.transcribe(
                    str(mp3),
                    language="hi",
                    word_timestamps=True,
                    condition_on_previous_text=False,
                    vad_filter=False,
                )

                segments = list(segments)

                lyrics = " ".join(
                    segment.text.strip()
                    for segment in segments
                    if segment.text.strip()
                )

                print(
                    f"[OK] Transcription: "
                    f"{len(lyrics)} characters",
                    flush=True,
                )

            except Exception as e:
                errors.append(
                    f"lyrics: {type(e).__name__}: {e}"
                )

                print(
                    f"[WHISPER ERROR] {e}",
                    flush=True,
                )

        # Always create a lyrics file, even if transcription fails.
        try:
            write_text(
                lyrics_file,
                (
                    lyrics
                    if lyrics
                    else "[LYRICS NOT GENERATED - CHECK LOG/JSON]"
                ),
            )

            print(
                f"[SAVED] {lyrics_file.name}",
                flush=True,
            )

        except Exception as e:
            errors.append(
                f"lyrics_file: {type(e).__name__}: {e}"
            )

            print(
                f"[LYRICS FILE ERROR] {e}",
                flush=True,
            )

        # ----------------------------------------------------
        # 4C-3. CALCULATE BPM AND KEY
        # ----------------------------------------------------

        try:
            chunk_audio = y[
                int(a * sr):int(b * sr)
            ]

            bpm, key = bpm_key(
                chunk_audio,
                sr,
            )

            print(
                f"[FEATURES] BPM={bpm}, key={key}",
                flush=True,
            )

        except Exception as e:
            errors.append(
                f"features: {type(e).__name__}: {e}"
            )

            print(
                f"[FEATURE ERROR] {e}",
                flush=True,
            )

        # ----------------------------------------------------
        # 4C-4. GENERATE THE GEMINI CAPTION
        # ----------------------------------------------------

        if audio_ok:

            try:
                print(
                    "[GEMINI] Requesting caption...",
                    flush=True,
                )

                # This wrapper calls the working gemini_caption()
                # function defined at the top of this cell.
                caption = caption_audio(
                    mp3,
                    genre,
                )

                if (
                    not isinstance(caption, str)
                    or not caption.strip()
                ):
                    raise RuntimeError(
                        "Caption function returned empty output"
                    )

                caption = caption.strip()

                print(
                    "[OK] Caption generated",
                    flush=True,
                )

            except Exception as e:
                errors.append(
                    f"caption: {type(e).__name__}: {e}"
                )

                print(
                    f"[CAPTION ERROR] {e}",
                    flush=True,
                )

        # Always create the caption file.
        try:
            write_text(
                caption_file,
                (
                    caption
                    if caption
                    else "[CAPTION NOT GENERATED - CHECK LOG/JSON]"
                ),
            )

            print(
                f"[SAVED] {caption_file.name}",
                flush=True,
            )

        except Exception as e:
            errors.append(
                f"caption_file: {type(e).__name__}: {e}"
            )

            print(
                f"[CAPTION FILE ERROR] {e}",
                flush=True,
            )

        # ----------------------------------------------------
        # 4C-5. SAVE JSON METADATA
        # ----------------------------------------------------

        status = (
            "complete"
            if not errors
            else "incomplete"
        )

        metadata = {
            "song": song.name,
            "genre": genre,
            "chunk": i,
            "audio_file": mp3.name,
            "caption": caption,
            "bpm": bpm,
            "keyscale": key,
            "timesignature": "4",
            "language": "hi",
            "start": a,
            "end": b,
            "lyrics": lyrics,
            "status": status,
            "errors": errors,
        }

        try:
            with open(
                json_file,
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    metadata,
                    f,
                    ensure_ascii=False,
                    indent=2,
                )

            print(
                f"[{status.upper()}] Saved metadata: "
                f"{json_file.name}",
                flush=True,
            )

        except Exception as e:
            song_errors += 1

            print(
                f"[JSON ERROR] {json_file}: {e}",
                flush=True,
            )

        # ----------------------------------------------------
        # 4C-6. REPORT CHUNK STATUS
        # ----------------------------------------------------

        if errors:

            song_errors += len(errors)

            print(
                f"[CHUNK INCOMPLETE] {base.name}: {errors}",
                flush=True,
            )

        else:
            print(
                f"[CHUNK COMPLETE] {base.name}",
                flush=True,
            )

    # --------------------------------------------------------
    # 4D. FINISH THIS SONG
    # --------------------------------------------------------

    print(
        f"\n[SONG FINISHED] {song.name} "
        f"| errors={song_errors}",
        flush=True,
    )

    return {
        "song": song.name,
        "chunks": total_chunks,
        "errors": song_errors,
    }


# ============================================================
# 5. RUN THE ONE-SONG TEST
# ============================================================

results = []

for index, song in enumerate(
    romantic_songs,
    start=1,
):

    print(
        f"\n========== SONG {index}/{len(romantic_songs)} ==========",
        flush=True,
    )

    try:
        result = process_song(song)

        results.append(result)

    except Exception as e:

        # Catch unexpected song-level errors so the run is reported.
        print(
            f"[UNEXPECTED SONG ERROR] {song.name}: "
            f"{type(e).__name__}: {e}",
            flush=True,
        )

        traceback.print_exc()

        results.append({
            "song": song.name,
            "chunks": 0,
            "errors": 1,
        })


# ============================================================
# 6. FINAL SUMMARY
# ============================================================

print(
    "\n========== FINAL SUMMARY ==========",
    flush=True,
)

print(
    f"Songs selected: {len(romantic_songs)}",
    flush=True,
)

print(
    f"Songs with errors: "
    f"{sum(r['errors'] > 0 for r in results)}",
    flush=True,
)

print(
    f"Total reported errors: "
    f"{sum(r['errors'] for r in results)}",
    flush=True,
)

print(
    f"Output directory: {DST.resolve()}",
    flush=True,
)

Found 1 song(s).
Input:  C:\Users\rdips\OneDrive\Documents\CS787proj\GACMIS\Dataset\Songs\Bollywood_Romantic
Output: C:\Users\rdips\OneDrive\Documents\CS787proj\GACMIS\Dataset\Chunks

========== SONG 1/1 ==========

[SONG] Abhijeet Srivastava - Chashni (From Bharat).opus
[AUDIO LOADED] Duration: 285.27 seconds
[CHUNKS] Detected 7

[CHUNK 1/7] 0.00s–39.01s
[FFMPEG] Extracting...
[OK] MP3: Abhijeet Srivastava - Chashni (From Bharat)_000.mp3
[WHISPER] Transcribing...
[OK] Transcription: 37 characters
[SAVED] Abhijeet Srivastava - Chashni (From Bharat)_000.lyrics.txt
[FEATURES] BPM=117, key=F# major
[GEMINI] Requesting caption...
[Gemini] Sending request (attempt 1/4): Abhijeet Srivastava - Chashni (From Bharat)_000.mp3
[Gemini] Caption generated successfully.
[OK] Caption generated
[SAVED] Abhijeet Srivastava - Chashni (From Bharat)_000.caption.txt
[COMPLETE] Saved metadata: Abhijeet Srivastava - Chashni (From Bharat)_000.json
[CHUNK COMPLETE] Abhijeet Srivastava - Chashni (From Bharat)_0

In [30]:
print("caption_audio" in globals())

False


In [ ]:

# ============================================================
# GACMIS PIPELINE — FIRST 25 BOLLYWOOD ROMANTIC SONGS
# MP3 chunks + Whisper lyrics + BPM/key + Gemini captions + JSON
# Resumable: completed chunks are skipped on subsequent runs.
# ============================================================

import json
import time
import subprocess
import tempfile
from pathlib import Path

import librosa
from google.genai import types


# -------------------- CONFIGURATION --------------------

SRC = Path(
    r"C:\Users\rdips\OneDrive\Documents\CS787proj"
    r"\GACMIS\Dataset\Songs"
)

DST = Path(
    r"C:\Users\rdips\OneDrive\Documents\CS787proj"
    r"\GACMIS\Dataset\Chunks"
)

ROMANTIC_DIR = SRC / "Bollywood_Romantic"

FFMPEG = (
    r"C:\Users\rdips\AppData\Local\Microsoft\WinGet\Packages"
    r"\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe"
    r"\ffmpeg-9.0.2-full_build\bin\ffmpeg.exe"
)

GEMINI_MODEL = "gemini-3.8-flash"
NUM_SONGS = 25
SKIP_EXISTING = True

AUDIO_EXTENSIONS = {
    ".mp3", ".opus", ".wav", ".m4a", ".flac", ".ogg"
}

# Keep your existing find_cuts(), bpm_key(), whisper, and client.
required_objects = ["find_cuts", "bpm_key", "whisper", "client"]
missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the notebook cells defining these first: "
        + ", ".join(missing_objects)
    )

if not Path(FFMPEG).is_file():
    raise FileNotFoundError(f"FFmpeg not found: {FFMPEG}")

if not ROMANTIC_DIR.is_dir():
    raise FileNotFoundError(
        f"Song directory not found: {ROMANTIC_DIR}"
    )

DST.mkdir(parents=True, exist_ok=True)


# -------------------- GEMINI CAPTION --------------------

def gemini_caption(path, prompt):
    """Generate a caption directly from audio bytes."""

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Audio file not found: {path}")

    mime_types = {
        ".mp3": "audio/mpeg",
        ".opus": "audio/ogg",
        ".wav": "audio/wav",
        ".m4a": "audio/mp4",
        ".flac": "audio/flac",
        ".ogg": "audio/ogg",
    }

    mime_type = mime_types.get(path.suffix.lower())

    if mime_type is None:
        raise ValueError(
            f"Unsupported audio format: {path.suffix}"
        )

    part = types.Part.from_bytes(
        data=path.read_bytes(),
        mime_type=mime_type,
    )

    last_error = None

    for attempt in range(4):
        try:
            print(
                f"[GEMINI] Attempt {attempt + 1}/4: {path.name}",
                flush=True,
            )

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=[part, prompt],
            )

            caption = (response.text or "").strip()

            if caption:
                return caption

            last_error = RuntimeError(
                "Gemini returned an empty caption."
            )

        except Exception as exc:
            last_error = exc
            print(
                f"[GEMINI ERROR] {type(exc).__name__}: {exc}",
                flush=True,
            )

        if attempt < 3:
            time.sleep(5 * (attempt + 1))

    raise RuntimeError(
        f"Gemini caption failed after 4 attempts: {path.name}"
    ) from last_error


def caption_audio(audio_path, genre):
    prompt = f"""
Describe this music segment from the genre/category: {genre}.

Write a concise but informative caption covering:
- Vocal presence and vocal style
- Language, if identifiable
- Instruments and accompaniment
- Rhythm, tempo feel, and mood
- Production and other audible musical characteristics

Describe only what is audible. Do not invent details.
Return only the caption in 2-3 sentences.
"""
    return gemini_caption(audio_path, prompt)


# -------------------- GENERAL HELPERS --------------------

def run_ffmpeg(args):
    """Run FFmpeg and raise an informative error if it fails."""

    command = [FFMPEG] + [str(x) for x in args]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    if result.returncode != 0:
        raise RuntimeError(
            "FFmpeg failed:\n"
            + result.stderr[-3000:]
        )


def retry_stage(stage_name, function, attempts=3):
    """Retry a pipeline stage; stop if all attempts fail."""

    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(
                f"[{stage_name}] Attempt {attempt}/{attempts}",
                flush=True,
            )
            return function()

        except Exception as exc:
            last_error = exc
            print(
                f"[{stage_name} ERROR] "
                f"{type(exc).__name__}: {exc}",
                flush=True,
            )

            if attempt < attempts:
                time.sleep(3 * attempt)

    raise RuntimeError(
        f"{stage_name} failed after {attempts} attempts."
    ) from last_error


def get_output_paths(output_dir, stem, chunk_index):
    base = output_dir / f"{stem}_{chunk_index:03d}"

    return {
        "base": base,
        "mp3": Path(str(base) + ".mp3"),
        "lyrics": Path(str(base) + ".lyrics.txt"),
        "caption": Path(str(base) + ".caption.txt"),
        "json": Path(str(base) + ".json"),
    }


def is_chunk_complete(paths):
    """Skip only if all required outputs and complete metadata exist."""

    try:
        required_files = [
            paths["mp3"],
            paths["lyrics"],
            paths["caption"],
            paths["json"],
        ]

        if not all(
            path.is_file() and path.stat().st_size > 0
            for path in required_files
        ):
            return False

        metadata = json.loads(
            paths["json"].read_text(encoding="utf-8")
        )

        return metadata.get("status") == "complete"

    except (OSError, ValueError, json.JSONDecodeError):
        return False


def save_json(path, data):
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def normalize_chunks(raw_chunks, duration):
    """
    Accept either [(start, end), ...] intervals or a list of
    cut-point times. The existing find_cuts algorithm is unchanged.
    """

    if raw_chunks is None:
        raise ValueError("find_cuts returned None.")

    raw_chunks = list(raw_chunks)

    if not raw_chunks:
        raise ValueError("find_cuts returned no chunks.")

    first = raw_chunks[0]

    if isinstance(first, (tuple, list)) and len(first) == 2:
        intervals = [
            (float(start), float(end))
            for start, end in raw_chunks
        ]
    else:
        points = [float(point) for point in raw_chunks]

        # Treat scalar values as cut points.
        if points[0] > 0.05:
            points.insert(0, 0.0)

        if points[-1] < duration:
            points.append(duration)

        intervals = list(zip(points[:-1], points[1:]))

    cleaned = []

    for start, end in intervals:
        start = max(0.0, start)
        end = min(duration, end)

        if end > start:
            cleaned.append((start, end))

    if not cleaned:
        raise ValueError("No valid chunk intervals were produced.")

    return cleaned


# -------------------- PROCESS ONE SONG --------------------

def process_song(song):
    print(f"\n[SONG] {song.name}", flush=True)

    genre = song.parent.name

    output_dir = DST / song.parent.relative_to(SRC)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Convert to a temporary WAV first, so librosa can load Opus
    # and other source formats consistently.
    temp_wav = None

    try:
        with tempfile.NamedTemporaryFile(
            suffix=".wav",
            delete=False,
        ) as temp:
            temp_wav = Path(temp.name)

        print("[AUDIO] Converting source to temporary WAV...", flush=True)

        retry_stage(
            "SOURCE CONVERSION",
            lambda: run_ffmpeg([
                "-y",
                "-i", song,
                "-vn",
                "-ac", "1",
                "-ar", "22050",
                "-c:a", "pcm_s16le",
                temp_wav,
            ]),
        )

        y, sr = librosa.load(
            str(temp_wav),
            sr=22050,
            mono=True,
        )

        duration = len(y) / sr

        print(
            f"[AUDIO LOADED] Duration: {duration:.2f} seconds",
            flush=True,
        )

        # Use your existing chunking algorithm without modifying it.
        raw_chunks = find_cuts(y, sr)
        chunks = normalize_chunks(raw_chunks, duration)

        print(f"[CHUNKS] Detected {len(chunks)}", flush=True)

        for chunk_index, (start, end) in enumerate(chunks):
            paths = get_output_paths(
                output_dir,
                song.stem,
                chunk_index,
            )

            base = paths["base"]
            mp3_path = paths["mp3"]
            lyrics_path = paths["lyrics"]
            caption_path = paths["caption"]
            json_path = paths["json"]

            print(
                f"\n[CHUNK {chunk_index + 1}/{len(chunks)}] "
                f"{start:.2f}s–{end:.2f}s",
                flush=True,
            )

            # Critical resume behavior: don't rerun completed chunks.
            if SKIP_EXISTING and is_chunk_complete(paths):
                print(
                    f"[SKIP] {base.name} is already complete.",
                    flush=True,
                )
                continue

            metadata = {
                "song": song.name,
                "genre": genre,
                "chunk_index": chunk_index,
                "audio_file": mp3_path.name,
                "caption": "",
                "bpm": None,
                "keyscale": None,
                "timesignature": "4",
                "language": "hi",
                "start": round(start, 3),
                "end": round(end, 3),
                "lyrics": "",
                "status": "incomplete",
                "errors": [],
            }

            try:
                # 1. Extract the chunk as MP3.
                def extract_chunk():
                    run_ffmpeg([
                        "-y",
                        "-ss", f"{start:.6f}",
                        "-i", song,
                        "-t", f"{end - start:.6f}",
                        "-vn",
                        "-ar", "44100",
                        "-b:a", "192k",
                        "-codec:a", "libmp3lame",
                        mp3_path,
                    ])

                    if (
                        not mp3_path.is_file()
                        or mp3_path.stat().st_size == 0
                    ):
                        raise RuntimeError(
                            "FFmpeg produced an empty MP3."
                        )

                retry_stage("FFMPEG EXTRACTION", extract_chunk)

                print(f"[OK] MP3: {mp3_path.name}", flush=True)

                # 2. Transcribe lyrics with Whisper.
                def transcribe_chunk():
                    result = whisper.transcribe(
                        str(mp3_path),
                        language="hi",
                        word_timestamps=True,
                        condition_on_previous_text=False,
                        vad_filter=False,
                    )

                    text = (result.get("text") or "").strip()

                    if not text:
                        raise RuntimeError(
                            "Whisper returned empty transcription."
                        )

                    return text

                lyrics = retry_stage(
                    "WHISPER TRANSCRIPTION",
                    transcribe_chunk,
                )

                lyrics_path.write_text(
                    lyrics + "\n",
                    encoding="utf-8",
                )

                metadata["lyrics"] = lyrics

                print(
                    f"[OK] Transcription: {len(lyrics)} characters",
                    flush=True,
                )

                # 3. Calculate BPM and key using your existing function.
                def calculate_features():
                    chunk_start = int(start * sr)
                    chunk_end = min(int(end * sr), len(y))
                    chunk_audio = y[chunk_start:chunk_end]

                    if len(chunk_audio) == 0:
                        raise ValueError(
                            "Empty audio array for this chunk."
                        )

                    return bpm_key(chunk_audio, sr)

                features = retry_stage(
                    "BPM/KEY ANALYSIS",
                    calculate_features,
                )

                # Expected return format: (bpm, keyscale).
                bpm, keyscale = features

                metadata["bpm"] = (
                    float(bpm)
                    if bpm is not None
                    else None
                )
                metadata["keyscale"] = (
                    str(keyscale)
                    if keyscale is not None
                    else None
                )

                print(
                    f"[FEATURES] BPM={bpm}, key={keyscale}",
                    flush=True,
                )

                # 4. Generate Gemini music caption.
                caption = retry_stage(
                    "GEMINI CAPTION",
                    lambda: caption_audio(mp3_path, genre),
                    attempts=2,
                )

                caption_path.write_text(
                    caption + "\n",
                    encoding="utf-8",
                )

                metadata["caption"] = caption

                print("[OK] Caption generated.", flush=True)

                # 5. Mark complete only after every stage succeeds.
                metadata["status"] = "complete"
                metadata["errors"] = []

                save_json(json_path, metadata)

                print(
                    f"[COMPLETE] Saved metadata: {json_path.name}",
                    flush=True,
                )

            except Exception as exc:
                metadata["status"] = "incomplete"
                metadata["errors"].append(
                    f"{type(exc).__name__}: {exc}"
                )

                # Persist the failure so it can be retried on rerun.
                save_json(json_path, metadata)

                print(
                    f"[STOP] Chunk failed: {base.name}",
                    flush=True,
                )
                print(
                    f"[ERROR] {type(exc).__name__}: {exc}",
                    flush=True,
                )
                print(
                    "[STOP] Pipeline halted. Fix the error and rerun "
                    "this cell; completed chunks will be skipped.",
                    flush=True,
                )

                # Do not silently move to the next chunk or song.
                return False

        print(f"\n[SONG COMPLETE] {song.name}", flush=True)
        return True

    finally:
        if temp_wav is not None and temp_wav.exists():
            try:
                temp_wav.unlink()
            except OSError:
                pass


# -------------------- RUN THE FIRST 25 SONGS --------------------

romantic_songs = sorted(
    (
        path
        for path in ROMANTIC_DIR.rglob("*")
        if path.is_file()
        and path.suffix.lower() in AUDIO_EXTENSIONS
    ),
    key=lambda path: str(path).lower(),
)[:NUM_SONGS]

print(f"Found {len(romantic_songs)} song(s).")
print(f"Input:  {ROMANTIC_DIR}")
print(f"Output: {DST}")
print(f"Skip completed chunks: {SKIP_EXISTING}")
print("=" * 60)

if not romantic_songs:
    raise FileNotFoundError(
        f"No supported audio files found in {ROMANTIC_DIR}"
    )

songs_completed_in_this_run = 0
songs_stopped_on_error = 0

for song_number, song in enumerate(romantic_songs, start=1):
    print(
        f"\n========== SONG {song_number}/{len(romantic_songs)} ==========",
        flush=True,
    )

    try:
        success = process_song(song)

    except Exception as exc:
        success = False
        print(
            f"[SONG ERROR] {song.name}: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )

    if success:
        songs_completed_in_this_run += 1
    else:
        songs_stopped_on_error += 1

        # Stop rather than moving past an error. Rerunning resumes
        # from incomplete work because completed chunks are skipped.
        break

print("\n" + "=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Songs selected:              {len(romantic_songs)}")
print(f"Songs completed this run:    {songs_completed_in_this_run}")
print(f"Songs stopped on error:      {songs_stopped_on_error}")
print(f"Output directory:            {DST}")


## 14. Inspect generated files

This lists a few output files and checks whether caption text is empty.

In [37]:
outputs = sorted(p for p in DST.rglob("*") if p.is_file())
print(f"Found {len(outputs)} output files under {DST.resolve()}")
for p in outputs[:30]:
    print(p.relative_to(DST), "-", p.stat().st_size, "bytes")

json_files = [p for p in outputs if p.suffix.lower() == ".json"]
empty_captions = []
for p in json_files:
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
        if not data.get("caption", "").strip():
            empty_captions.append(p)
    except Exception as e:
        print("Could not read JSON:", p, e)

print(f"JSON files: {len(json_files)}")
print(f"JSON files with empty captions: {len(empty_captions)}")
for p in empty_captions[:20]:
    print(" -", p)


Found 36 output files under C:\Users\rdips\OneDrive\Documents\CS787proj\GACMIS\Dataset\Chunks
_notebook_test\Aditi Singh Sharma - High Rated Gabru (Female Version)_test.mp3 - 974932 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_000.caption.txt - 393 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_000.json - 886 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_000.lyrics.txt - 95 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_000.mp3 - 937316 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_001.caption.txt - 433 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_001.json - 1411 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_001.lyrics.txt - 567 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_001.mp3 - 988725 bytes
Bollywood_Romantic\Abhijeet Srivastava - Chashni (From Bharat)_002.caption.txt - 433 bytes
Bollywood_Romantic\